# 🦺 PPE Violation Detector — Mentor Demo
**Run this notebook AFTER training is complete.**

This notebook gives you a live web app where you (or your mentor) can:
- **Upload any workplace image**
- Get **bounding boxes** drawn around each person and PPE item
- See a clear **VIOLATION ⚠ / COMPLIANT ✅** banner
- Read a **detection report** (which PPE is missing / worn wrong)

> 🔗 A **public shareable link** is generated — paste it in chat/email so your mentor can try it from their own browser.

---
## ✅ 0 · Setup

In [ ]:
# Verify GPU first
!nvidia-smi | head -12

In [ ]:
# Install / upgrade required packages
!pip install ultralytics gradio opencv-python-headless -q

import torch, ultralytics, gradio
print('PyTorch    :', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('Ultralytics:', ultralytics.__version__)
print('Gradio     :', gradio.__version__)

---
## 📂 1 · Clone / Refresh Project Repo

In [ ]:
import os

PROJECT_DIR = '/content/gril_assessment_3'
REPO_URL    = 'https://github.com/Amitesh-Verma003/gril_assessment_3.git'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
!ls

---
## 🏋️ 2 · Point to Your Trained Weights

**Option A** – Upload `best.pt` from your PC (easiest):  
Run the cell below and use the file-picker.

**Option B** – Already trained in this Colab session:  
Skip to the next cell — weights are already at `runs/train/ppe_yolov8n/weights/best.pt`.

**Option C** – Download from Google Drive:  
Mount Drive and set `WEIGHTS_PATH` accordingly.

In [ ]:
# ── Option A: Upload best.pt from your PC ──────────────────
from google.colab import files
print('Upload your best.pt weight file:')
uploaded = files.upload()
WEIGHTS_PATH = list(uploaded.keys())[0]
print(f'Loaded: {WEIGHTS_PATH}')

In [ ]:
# ── Option B: use weights from training run ────────────────
# Change model variant if needed (n / s / m / l)
MODEL_VARIANT = 'yolov8n'
WEIGHTS_PATH  = f'runs/train/ppe_{MODEL_VARIANT}/weights/best.pt'

import os
if os.path.exists(WEIGHTS_PATH):
    print(f'✅ Found: {WEIGHTS_PATH}')
else:
    print(f'❌ Not found: {WEIGHTS_PATH}')
    print('   Run the training notebook first, or use Option A above.')

---
## 🔍 3 · Quick Single-Image Test (no UI needed)
Good for a fast sanity check before launching the full app.

In [ ]:
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

CLASS_NAMES = ['person','hardhat','hardhat_wrong','no_hardhat',
               'vest','vest_wrong','no_vest','gloves','no_gloves']
VIOLATION_IDS = {2,3,5,6,8}
VIOLATION_LABELS = {2:'hardhat_wrong',3:'no_hardhat',
                    5:'vest_wrong',   6:'no_vest', 8:'no_gloves'}
BOX_COLORS = {
    0:(0,165,255), 1:(0,200,0), 2:(0,0,220), 3:(0,0,220),
    4:(0,200,0),   5:(50,50,220),6:(0,0,220), 7:(0,200,0), 8:(0,0,220)
}

model = YOLO(WEIGHTS_PATH)

# pick a random test image
test_imgs = list(Path('dataset/images/test').glob('*.*'))
if not test_imgs:
    test_imgs = list(Path('dataset/images/val').glob('*.*'))
img_path = random.choice(test_imgs)
print(f'Testing on: {img_path.name}')

result  = model.predict(str(img_path), conf=0.4, iou=0.45, verbose=False)[0]
img_bgr = cv2.imread(str(img_path))
out     = img_bgr.copy()
h, w    = out.shape[:2]
violations = []

for box, cls_id, conf in zip(
    result.boxes.xyxy.cpu().numpy(),
    result.boxes.cls.int().cpu().tolist(),
    result.boxes.conf.cpu().tolist()
):
    x1,y1,x2,y2 = map(int,box)
    c = BOX_COLORS.get(cls_id,(180,180,180))
    lbl = f'{CLASS_NAMES[cls_id]} {conf:.2f}'
    cv2.rectangle(out,(x1,y1),(x2,y2),c,2)
    (tw,th),_ = cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.55,1)
    cv2.rectangle(out,(x1,y1-th-8),(x1+tw+4,y1),c,-1)
    cv2.putText(out,lbl,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1,cv2.LINE_AA)
    if cls_id in VIOLATION_IDS: violations.append(VIOLATION_LABELS[cls_id])

# banner
if violations:
    cv2.rectangle(out,(0,0),(w,36),(0,0,180),-1)
    cv2.putText(out,f'VIOLATION: {", ".join(set(violations))}',(10,25),
                cv2.FONT_HERSHEY_SIMPLEX,0.75,(255,255,255),2,cv2.LINE_AA)
else:
    cv2.rectangle(out,(0,0),(w,36),(0,150,0),-1)
    cv2.putText(out,'ALL PPE COMPLIANT',(10,25),
                cv2.FONT_HERSHEY_SIMPLEX,0.75,(255,255,255),2,cv2.LINE_AA)

plt.figure(figsize=(10,8))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.title(f'File: {img_path.name} | Violations: {list(set(violations)) or "None"}')
plt.axis('off'); plt.tight_layout(); plt.show()

---
## 🌐 4 · Launch Gradio Web App (Share with Mentor)

Running this cell will print a **public URL** like:
```
Running on public URL: https://xxxxxxxx.gradio.live
```
Copy that link and share it with your mentor — they open it in any browser.

> The link stays alive as long as this Colab session is running (up to 72 hrs on Gradio free tier).

In [ ]:
# ── Patch the weights path so app.py picks up your model ──
import os
os.environ['PPE_WEIGHTS'] = WEIGHTS_PATH

# Update DEFAULT_WEIGHTS in app.py at runtime
app_src = open('app.py').read()
app_src = app_src.replace(
    'DEFAULT_WEIGHTS = "runs/train/ppe_yolov8n/weights/best.pt"',
    f'DEFAULT_WEIGHTS = "{WEIGHTS_PATH}"'
)
open('app_runtime.py','w').write(app_src)
print('Patched weights path →', WEIGHTS_PATH)

In [ ]:
# ── Launch the Gradio app with public share link ───────────
exec(open('app_runtime.py').read())

---
## 📸 5 · Batch Test: Run on All Test Images & Save Grid
Useful for showing your mentor a visual summary of model performance.

In [ ]:
import cv2, matplotlib.pyplot as plt
from pathlib import Path
import math, random

N_SHOW   = 12       # number of images to show in the grid
COLS     = 4
CONF     = 0.4
IOU      = 0.45

test_imgs = random.sample(list(Path('dataset/images/test').glob('*.*')),
                           min(N_SHOW, len(list(Path('dataset/images/test').glob('*.*')))))
preds     = model.predict(test_imgs, conf=CONF, iou=IOU, imgsz=640,
                           verbose=False)

rows_n = math.ceil(len(preds) / COLS)
fig, axes = plt.subplots(rows_n, COLS, figsize=(6*COLS, 5*rows_n))
axes = axes.flat

summary = {'compliant': 0, 'violation': 0}

for ax, r in zip(axes, preds):
    img = cv2.imread(r.path)
    h, w = img.shape[:2]
    cls_ids   = r.boxes.cls.int().cpu().tolist()
    violations= [VIOLATION_LABELS[c] for c in cls_ids if c in VIOLATION_IDS]

    for box, cls_id, conf in zip(
        r.boxes.xyxy.cpu().numpy(),
        cls_ids, r.boxes.conf.cpu().tolist()
    ):
        x1,y1,x2,y2 = map(int,box)
        c = BOX_COLORS.get(cls_id,(180,180,180))
        cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
        cv2.putText(img,f'{CLASS_NAMES[cls_id]} {conf:.2f}',(x1,max(y1-5,15)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.45,c,1,cv2.LINE_AA)

    if violations:
        cv2.rectangle(img,(0,0),(w,30),(0,0,160),-1)
        cv2.putText(img,'VIOLATION',(8,21),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
        summary['violation'] += 1
        t_color = 'red'
    else:
        cv2.rectangle(img,(0,0),(w,30),(0,120,0),-1)
        cv2.putText(img,'COMPLIANT',(8,21),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
        summary['compliant'] += 1
        t_color = 'green'

    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title('\n'.join(set(violations)) or 'OK', fontsize=7, color=t_color)
    ax.axis('off')

for ax in axes:  ax.axis('off')   # hide empty cells

plt.suptitle(
    f'PPE Detection — {N_SHOW} Test Images   '
    f'| ✅ Compliant: {summary["compliant"]}  '
    f'| ⚠ Violations: {summary["violation"]}',
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.savefig('outputs/mentor_demo_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grid saved → outputs/mentor_demo_grid.png')

In [ ]:
# Download the grid image to your PC
from google.colab import files
files.download('outputs/mentor_demo_grid.png')